# 2. Training

`train.py` trains every model of the paper; `options.py` holds the defaults. With the released
data, the seed of the paper, the package versions of `requirements.txt` and the same GPU type,
the trained `model.pt` is byte-identical to the released checkpoint. The full runs take from
about 10 minutes (N2N) to a few hours (Noise2Void, the 270 s models) on one GPU.

| model of the paper | command |
|---|---|
| N2N, mixed-sigma (the model of the paper) | `python train.py --model n2n --seed 1` |
| the other four training runs | `python train.py --model n2n --seed 2` (3, 4, 5) |
| N2N without augmentation | `python train.py --model n2n --noise-aug-max 0 --seed 0` |
| 90 / 135 / 270 s pairs | `python train.py --model n2n --pair-gap 2 --align 1 --seed 0` (3; 6 with seeds 0, 2, 3) |
| depth 1 / 3, width 16 / 64 | `python train.py --model n2n --levels 1 --seed 0` (`--levels 3`, `--base 16`, `--base 64`) |
| Noise2Void | `python train.py --model n2v` |
| supervised U-Net, without / with mixed-sigma | `python train.py --model supervised` / `... --noise-aug-max 21` |
| Park et al. (2020) | `python train.py --model park2020` |

Every command also takes `--data-root <data> --out-dir <folder>`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")          # the repository root, one level up from notebooks/
DATA_ROOT = Path("../data")       # the public data: manifests/, frames/, stacks/, pairs/, checkpoints/, results/
import hashlib, subprocess
import options
for model, d in options.DEFAULTS.items():
    print(model, d)

## A short run (a few hundred steps) to check that everything works

In [ ]:
RUN = Path("runs/demo")
subprocess.run([sys.executable, "../train.py", "--model", "n2n", "--data-root", str(DATA_ROOT),
                "--out-dir", str(RUN), "--steps", "200", "--val-every", "100"], check=True)
print(open(RUN / "metrics.jsonl").read()[-400:])

## Compare a checkpoint with the released one

A full run (`--steps` left at 20,000) writes `model.pt`; compare its SHA-256 with
`checkpoints/SHA256SUMS`. Keep the file name `model.pt`: the name is stored inside the file.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

released = {line.split()[1]: line.split()[0] for line in open(DATA_ROOT / "checkpoints" / "SHA256SUMS")}
mine = Path("runs/n2n_mixsigma_s1/model.pt")        # the output folder of your full run
if mine.exists():
    print("identical to the released checkpoint:", sha256(mine) == released["n2n_mixsigma_s1/model.pt"])